In [ ]:
import cv2
import time
from threading import Thread
import platform
import os
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import numpy as np

# --- ✨ NEW: Import for the virtual camera ---
import pyvirtualcam

# Wrap deepface import
try:
    from deepface import DeepFace
except ImportError:
    print("="*80); print("ERROR: DeepFace not found."); print("Please install it: !pip install deepface"); print("="*80); raise

# --- Configuration and Core Functions (All unchanged from before) ---
ALERT_THRESHOLD_SECONDS = 3.0
EMOTION_ANALYSIS_INTERVAL_FRAMES = 15
GRAPH_UPDATE_INTERVAL_FRAMES = 10
FONT = cv2.FONT_HERSHEY_SIMPLEX
COLOR_ATTENTIVE = (0, 255, 0); COLOR_DISTRACTED = (0, 0, 255)
COLOR_TEXT_BG = (0, 0, 0); COLOR_TEXT = (255, 255, 255)

last_emotion_analysis = {}; is_analyzing_emotion = False; attention_log = []

def play_alert_sound():
    print("ALERT! User is distracted.")
    system_name = platform.system()
    try:
        if system_name == "Windows": import winsound; winsound.Beep(1000, 500)
        elif system_name == "Darwin": os.system("afplay /System/Library/Sounds/Glass.aiff")
        else: os.system("play -nq -t alsa synth 0.5 sine 1000")
    except Exception: print('\a')

def analyze_emotions(face_roi):
    global last_emotion_analysis, is_analyzing_emotion
    try:
        result = DeepFace.analyze(face_roi, actions=['emotion'], enforce_detection=False)
        if isinstance(result, list) and len(result) > 0: last_emotion_analysis = result[0].get('emotion', {})
    finally: is_analyzing_emotion = False

def draw_emotion_data(frame, face_coords):
    if not last_emotion_analysis: return
    (x, y, w, h) = face_coords; overlay = frame.copy()
    bg_y_start = y + h + 10; num_emotions = len(last_emotion_analysis)
    bg_y_end = bg_y_start + (num_emotions * 22) + 10
    cv2.rectangle(overlay, (x, bg_y_start), (x + w, bg_y_end), COLOR_TEXT_BG, -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    for i, (emotion, percentage) in enumerate(last_emotion_analysis.items()):
        text = f"{emotion.capitalize()}: {percentage:.1f}%"
        text_y = bg_y_start + (i + 1) * 20
        cv2.putText(frame, text, (x + 5, text_y), FONT, 0.5, COLOR_TEXT, 1, cv2.LINE_AA)

def update_graphs(ax1, ax2):
    ax1.clear(); ax2.clear()
    if last_emotion_analysis:
        emotions = list(last_emotion_analysis.keys()); percentages = list(last_emotion_analysis.values())
        ax1.bar(emotions, percentages, color='skyblue'); ax1.set_title("Live Emotion Analysis")
        ax1.set_ylabel("Confidence (%)"); ax1.set_ylim(0, 100)
    else:
        ax1.set_title("Live Emotion Analysis"); ax1.text(0.5, 0.5, "Awaiting face detection...", ha='center', va='center')
    if attention_log:
        attentive_time = attention_log.count('attentive'); distracted_time = attention_log.count('distracted')
        labels = ['Attentive', 'Distracted']; sizes = [attentive_time, distracted_time]
        colors = ['#90ee90', '#ffcccb']
        ax2.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90); ax2.axis('equal')
    ax2.set_title("Attention Session Summary")

# --- Main Execution Block ---
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
if not face_cascade.empty():
    video_capture = cv2.VideoCapture(0)
    if video_capture.isOpened():
        # ✨ NEW: Get webcam dimensions for the virtual camera
        width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = video_capture.get(cv2.CAP_PROP_FPS)

        print("Webcam started successfully...")
        print(">>> IMPORTANT: Press 'q' in the video window to stop. <<<")

        plt.ion(); fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        fig.canvas.manager.set_window_title('Real-Time Analytics Dashboard')
        
        distraction_start_time = None; alert_triggered = False; frame_counter = 0

        # --- ✨ NEW: Setup and start the virtual camera ---
        with pyvirtualcam.Camera(width=width, height=height, fps=fps) as cam:
            print(f'Virtual camera started ({cam.device}). Select this in your meeting software.')
            try:
                while True:
                    ret, frame = video_capture.read()
                    if not ret: break

                    frame = cv2.flip(frame, 1)
                    # The core detection logic is COMPLETELY UNCHANGED
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))
                    
                    current_status = 'distracted'
                    if len(faces) > 0:
                        current_status = 'attentive'
                        # (All the attentive state logic is the same...)
                        distraction_start_time = None
                        if alert_triggered: print("User is attentive again."); alert_triggered = False
                        (x, y, w, h) = faces[0]
                        cv2.rectangle(frame, (x, y), (x + w, y + h), COLOR_ATTENTIVE, 2)
                        cv2.putText(frame, "Status: Attentive", (10, 30), FONT, 0.7, COLOR_ATTENTIVE, 2, cv2.LINE_AA)
                        if frame_counter % EMOTION_ANALYSIS_INTERVAL_FRAMES == 0 and not is_analyzing_emotion:
                            is_analyzing_emotion = True
                            face_roi = frame[y:y+h, x:x+w]
                            analysis_thread = Thread(target=analyze_emotions, args=(face_roi,))
                            analysis_thread.start()
                        draw_emotion_data(frame, faces[0])
                    else:
                        # (All the distracted state logic is the same...)
                        if distraction_start_time is None: distraction_start_time = time.time()
                        elapsed_time = time.time() - distraction_start_time
                        remaining_time = max(0, ALERT_THRESHOLD_SECONDS - elapsed_time)
                        cv2.putText(frame, f"Status: Distracted ({remaining_time:.1f}s)", (10, 30), FONT, 0.7, COLOR_DISTRACTED, 2, cv2.LINE_AA)
                        if elapsed_time > ALERT_THRESHOLD_SECONDS and not alert_triggered:
                            alert_triggered = True; play_alert_sound()
                    
                    frame_counter += 1; attention_log.append(current_status)
                    if frame_counter % GRAPH_UPDATE_INTERVAL_FRAMES == 0: update_graphs(ax1, ax2)

                    # --- ✨ NEW: Send the processed frame to the virtual camera ---
                    # Convert frame from OpenCV's BGR to RGB format, which pyvirtualcam expects
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    cam.send(frame_rgb)
                    cam.sleep_until_next_frame()

                    cv2.imshow('Live Feed (Local Preview) - Press "q" to Quit', frame)
                    plt.pause(0.001)
                    if cv2.waitKey(1) & 0xFF == ord('q'): break
            finally:
                video_capture.release(); cv2.destroyAllWindows()
                plt.ioff(); plt.close()
                print("Webcam released and all windows closed.")

In [ ]:
import tensorflow as tf
print(tf.__version__)
from deepface import DeepFace
print("DeepFace imported successfully!")


In [ ]:
pip install matplotlib


In [ ]:
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
print("Matplotlib loaded successfully!")


In [ ]:
pip install pyvirtualcam


In [ ]:
import pyvirtualcam
print("pyvirtualcam imported successfully!")


In [ ]:
pip install tf-keras


In [ ]:
from deepface import DeepFace
print("DeepFace imported successfully!")
